### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so I want to figure out why parrots talk. Let me start by recalling what I know. Parrots are known for their ability to mimic human speech. I remember seeing parrots on TV or in movies that can say words like "hello" or "goodbye." But why do they do that? Is it just imitation, or is there more to it?\n\nFirst, I should consider their natural behavior. Parrots are social birds. In the wild, they live in flocks and communicate with each other through calls. Maybe talking is an extension of that social behavior. If they can mimic human sounds, it might help them integrate into human social groups. That makes sense because humans are social too, so the parrot might be trying to fit in.\n\nAnother thought: parrots have a syrinx, which is their vocal organ, and it\'s different from human vocal cords. This allows them to produce a wide range of sounds. But how does that relate to talking? Maybe their syrinx is more flexible, enabling them to mimic various sou

In [4]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    # In a real implementation, this would call a weather API.
    return f"The current weather in {location} is sunny with a high of 25°C."

model_with_tools = model.bind_tools([get_weather]) # or you can use create_agent to create an agent with tools

In [9]:
response = model_with_tools.invoke("What's the weather like in New York?")
print(response)

for tool_call in response.tool_calls:
    print(f"Tool called: {tool_call['name']} with arguments {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in New York. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is provided as "New York". So I\'ll call the function with that location. Make sure the JSON is correctly formatted with the name and arguments.\n', 'tool_calls': [{'id': 'fe251m2n8', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 90, 'prompt_tokens': 157, 'total_tokens': 247, 'completion_time': 0.148164143, 'completion_tokens_details': {'reasoning_tokens': 65}, 'prompt_time': 0.006098295, 'prompt_tokens_details': None, 'queue_time': 0.022257305, 'total_time': 0.154262438}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_d58dbe76cd', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d0b1d-c

### Tool Execution Loop

In [10]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The current weather in Boston is sunny with a high of 25°C. Let me know if you'd like more details! ☀️


In [11]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools provided. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call this function with "Boston" as the location. I\'ll make sure to format the tool call correctly within the XML tags as instructed.\n', 'tool_calls': [{'id': '26trg2shs', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 155, 'total_tokens': 248, 'completion_time': 0.141462375, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.007817866, 'prompt_tokens_details': None, 'queue_time': 0.023292492, 'total_time': 0.149280241}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_d58dbe76cd', 'service_tier': 'on_demand